In [2]:
import pandas as pd
import numpy as np
from scipy.constants import pi

branch_data = pd.read_csv("branch.csv")
h_5 = 5 * 50

def create_z_5(branch_data):
    return [
        complex(row['r'], row['l'] * 2 * pi * h_5)
        for _, row in branch_data.iterrows()
    ]

def create_y_5(branch_data):
    return [
        complex(0, row['c'] * 2 * pi * h_5)
        for _, row in branch_data.iterrows()
    ]

def create_gammaL_5(branch_data, z_5, y_5):
    z_5 = np.array(z_5)
    y_5 = np.array(y_5)
    gammaL_5 = []
    for i, row in branch_data.iterrows():
        length = row['Length']
        gammaL = length * np.sqrt(z_5[i] * y_5[i])
        gammaL_5.append(gammaL)
    return gammaL_5

z_5 = create_z_5(branch_data)
y_5 = create_y_5(branch_data)
gammaL_5 = create_gammaL_5(branch_data, z_5, y_5)

def Z0_5(z_5, y_5):
    return np.sqrt(np.array(z_5) / np.array(y_5))

Z0_5 = Z0_5(z_5, y_5)
def ABCD_5(gammaL_5, Z0_5):
    A = np.cosh(gammaL_5)
    B = Z0_5 * np.sinh(gammaL_5)
    C = (1 / Z0_5) * np.sinh(gammaL_5)
    D = np.cosh(gammaL_5)
    return A, B, C, D
A_5, B_5, C_5, D_5 = ABCD_5(gammaL_5, Z0_5)
ABCD_5_df = pd.DataFrame({
    'From Bus Number': branch_data['From Bus  Number'],
    'To Bus Number'  : branch_data['To Bus  Number'],
    'A_5'           : A_5,
    'B_5'           : B_5,
    'C_5'           : C_5,
    'D_5'           : D_5
})

ABCD_5_df.to_csv("ABCD_5.csv", index=False)


In [ ]:
import pandas as pd
import numpy as np
from numpy.linalg import pinv

def calculate_all_equivalent_inductances(csv_path):
    # System Constants
    Z_BASE = 484.0  # Ohms
    F = 50.0        # Hz
    OMEGA = 2 * np.pi * F  # ~314.159 rad/s
    
    # Target locations to evaluate mapped to CSV substring
    target_locations = {
        'Puttalam': 'PUTTALAM',
        'New Anuradhapura': 'NEWANU',
        'Vavuniya': 'VAVUNIYA',
        'Mannar': 'MANNAR',
        'Nadukuda': 'NADUKUDA',
        'New Chilaw': 'NCHILAW',
        'Veyangoda': 'VEYAN',
        'Kotugoda': 'KOTUG',
        'New Kerawalapitiya 2': 'NEW_KERA2',
        'New Kerawalapitiya': 'NEW_KERA', # Note: May also capture KERA2 due to substring match
        'Kerawalapitiya': 'KERAWALA',
        'Colombo-L': 'COL_L',
        'Biyagama': 'BIYAG',
        'Kelaniya': 'KELAN',
        'Barge': 'BARGE',
        'Pannipitiya': 'PANNI-2',
        'Panni D2': 'PANNI-D',
        'Padukka': 'PADUKKA',
        'New Polpitiya': 'NPOLP',
        'Hambantota': 'HAMBAN',
        'Kotmale': 'KOTMA',
        'Upper Kotmale': 'UPPER-KOTH',
        'Victoria': 'VICTO',
        'Randenigala': 'RANDE',
        'Rantambe': 'RANTE',
        'New Habarana': 'NHAB'
    }

    # 1. Load and clean the data
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: Could not find {csv_path}.")
        return

    # Clean headers and string columns
    df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)
    df['From Bus Name'] = df['From Bus Name'].astype(str).str.strip()
    df['To Bus Name'] = df['To Bus Name'].astype(str).str.strip()

    # 2. Extract all unique buses to build our matrix indices
    from_buses = df[['From Bus Number', 'From Bus Name']].rename(columns={'From Bus Number': 'BusNum', 'From Bus Name': 'BusName'})
    to_buses = df[['To Bus Number', 'To Bus Name']].rename(columns={'To Bus Number': 'BusNum', 'To Bus Name': 'BusName'})
    all_buses = pd.concat([from_buses, to_buses]).drop_duplicates(subset=['BusNum']).reset_index(drop=True)
    
    # Map bus numbers to matrix indices (0 to N-1)
    bus_to_idx = {row['BusNum']: idx for idx, row in all_buses.iterrows()}
    n_buses = len(all_buses)

    # 3. Build the Admittance Matrix (Y-bus)
    Y_bus = np.zeros((n_buses, n_buses), dtype=complex)

    for _, row in df.iterrows():
        i = bus_to_idx[row['From Bus Number']]
        j = bus_to_idx[row['To Bus Number']]
        
        # Line parameters
        r = row['Line R (pu)']
        x = row['Line X (pu)']
        b = row['Charging B (pu)']
        
        # Calculate Series Admittance (y = 1 / z)
        z_series = complex(r, x)
        y_series = 1.0 / z_series if z_series != 0 else 0
        
        # Calculate Shunt Admittance (Line charging is split between both ends)
        y_shunt = complex(0, b / 2.0)
        
        # Populate diagonals (Self-admittance)
        Y_bus[i, i] += y_series + y_shunt
        Y_bus[j, j] += y_series + y_shunt
        
        # Populate off-diagonals (Mutual admittance)
        Y_bus[i, j] -= y_series
        Y_bus[j, i] -= y_series

    # 4. Invert Y-bus to get Impedance Matrix (Z-bus)
    print("Inverting network matrix to calculate equivalents...\n")
    Z_bus = pinv(Y_bus)

    # 5. Extract and print results for all target locations
    print("=" * 70)
    print(f"{'Location':<25} | {'Bus Number':<12} | {'Eq. Inductance (mH)':<20}")
    print("=" * 70)

    # Keep track of printed buses to avoid duplicate lines if substring matches overlap
    printed_buses = set()

    for display_name, csv_name in target_locations.items():
        # Find the bus in the network
        target_bus = all_buses[all_buses['BusName'].str.upper().str.contains(csv_name.upper())]
        
        if target_bus.empty:
            print(f"{display_name:<25} | {'NOT FOUND':<12} | {'N/A':<20}")
            continue
        
        # Loop over matches (in case a location has multiple buses like 220kV vs 132kV)
        for _, row in target_bus.iterrows():
            bus_num = row['BusNum']
            
            # Skip if we already printed this exact bus (handles overlap like NEW_KERA and NEW_KERA2)
            if bus_num in printed_buses:
                continue
                
            printed_buses.add(bus_num)
            idx = bus_to_idx[bus_num]
            
            # Extract diagonal element from Z-bus
            z_eq_pu = Z_bus[idx, idx]
            
            # Convert reactance to inductance
            x_eq_ohm = z_eq_pu.imag * Z_BASE
            l_eq_mh = (x_eq_ohm / OMEGA) * 1000
            
            print(f"{display_name:<25} | {bus_num:<12} | {l_eq_mh:.2f} mH")

    print("=" * 70)
    print("* NOTE: This represents the transmission network equivalent ONLY.")

if __name__ == "__main__":
    calculate_all_equivalent_inductances('branch.csv')

Inverting network matrix to calculate equivalents...

Location                  | Bus Number   | Eq. Inductance (mH) 
Puttalam                  | 2810         | -346.41 mH
New Anuradhapura          | 2705         | -342.36 mH
Vavuniya                  | 2245         | -320.55 mH
Mannar                    | 2280         | -281.14 mH
Nadukuda                  | 2281         | -266.84 mH
New Chilaw                | 2815         | -353.99 mH
Veyangoda                 | 2830         | -360.39 mH
Kotugoda                  | 2580         | -363.29 mH
New Kerawalapitiya 2      | 2306         | -356.68 mH
New Kerawalapitiya        | 2307         | -354.66 mH
Kerawalapitiya            | 2305         | -356.55 mH
Colombo-L                 | 2350         | -348.64 mH
Biyagama                  | 2570         | -364.37 mH
Kelaniya                  | 2300         | -358.74 mH
Barge                     | 2222         | -357.83 mH
Pannipitiya               | 2560         | -349.91 mH
Panni D2          

In [1]:
# UGC per-unit-length parameters (220 kV XLPE cable)
c_ugc = 1e-7     # F/km (capacitance per unit length)

# Section length
L_ugc = 6.0         # km (length of the UGC section)

# Calculate Total Capacitance
total_capacitance = c_ugc * L_ugc

# Display the results
print("--- UGC Capacitance Calculation ---")
print(f"Cable Length: {L_ugc} km")
print(f"Capacitance per km: {c_ugc} F/km")
print("-" * 33)
print(f"Total Capacitance: {total_capacitance} F")
print(f"Total Capacitance: {total_capacitance * 1e6:.2f} µF (Microfarads)")

--- UGC Capacitance Calculation ---
Cable Length: 6.0 km
Capacitance per km: 1e-07 F/km
---------------------------------
Total Capacitance: 6e-07 F
Total Capacitance: 0.60 µF (Microfarads)


In [20]:
import numpy as np

# ---------------------------------------------------------
# 1. Input Parameters
# ---------------------------------------------------------
# Total capacitance from the 8 km 220kV XLPE UGC calculation
C_ugc_F = 6e-7  # 2.0 µF in Farads

# Equivalent grid inductances (Absolute values from Z-bus inversion)
grid_inductances_mH = {
    'Puttalam': 346.41,
    'New Anuradhapura': 342.36,
    'Vavuniya': 320.55,
    'Mannar': 281.14,
    'Nadukuda': 266.84,
    'New Chilaw': 353.99,
    'Veyangoda': 360.39,
    'Kotugoda': 363.29,
    'New Kerawalapitiya 2': 356.68,
    'New Kerawalapitiya': 354.66,
    'Kerawalapitiya': 356.55,
    'Colombo-L': 348.64,
    'Biyagama': 364.37,
    'Kelaniya': 358.74,
    'Barge': 357.83,
    'Pannipitiya': 349.91,
    'Panni D2': 358.51,
    'Padukka': 357.00,
    'New Polpitiya': 360.78,
    'Hambantota': 296.97,
    'Kotmale': 362.95,
    'Upper Kotmale': 351.70,
    'Victoria': 348.95,
    'Randenigala': 333.13,
    'Rantambe': 330.11,
    'New Habarana': 340.03
}

# ---------------------------------------------------------
# 2. Calculation and Output
# ---------------------------------------------------------
print("=" * 65)
print(f"Resonance Frequency Calculation (C = {C_ugc_F * 1e6:.1f} µF)")
print("=" * 65)
print(f"{'Location':<25} | {'|L| (mH)':<10} | {'Resonance Freq (Hz)':<20}")
print("-" * 65)

for location, L_mH in grid_inductances_mH.items():
    # Convert mH to Henrys
    L_H = L_mH * 1e-3
    
    # Calculate resonance frequency: f_r = 1 / (2 * pi * sqrt(L * C))
    f_r = 1 / (2 * np.pi * np.sqrt(L_H * C_ugc_F))
    
    # Print the formatted row
    print(f"{location:<25} | {L_mH:<10.2f} | {f_r:.2f} Hz")

print("=" * 65)

Resonance Frequency Calculation (C = 0.6 µF)
Location                  | |L| (mH)   | Resonance Freq (Hz) 
-----------------------------------------------------------------
Puttalam                  | 346.41     | 349.10 Hz
New Anuradhapura          | 342.36     | 351.16 Hz
Vavuniya                  | 320.55     | 362.91 Hz
Mannar                    | 281.14     | 387.51 Hz
Nadukuda                  | 266.84     | 397.76 Hz
New Chilaw                | 353.99     | 345.34 Hz
Veyangoda                 | 360.39     | 342.26 Hz
Kotugoda                  | 363.29     | 340.89 Hz
New Kerawalapitiya 2      | 356.68     | 344.04 Hz
New Kerawalapitiya        | 354.66     | 345.02 Hz
Kerawalapitiya            | 356.55     | 344.10 Hz
Colombo-L                 | 348.64     | 347.98 Hz
Biyagama                  | 364.37     | 340.39 Hz
Kelaniya                  | 358.74     | 343.05 Hz
Barge                     | 357.83     | 343.48 Hz
Pannipitiya               | 349.91     | 347.35 Hz
Panni D2   

In [11]:
import pandas as pd
import numpy as np

# Load and ensure numeric types
ABCD = pd.read_csv("ABCD_5.csv")

# Clean column names (optional, in case there are hidden spaces)
ABCD.columns = ABCD.columns.str.strip()

# Convert to complex numbers safely
for col in ['A_5', 'B_5', 'C_5', 'D_5']:
    ABCD[col] = ABCD[col].apply(lambda x: complex(x.replace('i', 'j')) if isinstance(x, str) else complex(x))

def combine_parallel(group):
    if len(group) == 1:
        row = group.iloc[0]
        return pd.Series({
            'A': row['A_5'],
            'B': row['B_5'],
            'C': row['C_5'],
            'D': row['D_5']
        })

    # Start with first line
    A_eq, B_eq, C_eq, D_eq = group.iloc[0][['A_5', 'B_5', 'C_5', 'D_5']]
    for _, row in group.iloc[1:].iterrows():
        A1, B1, C1, D1 = A_eq, B_eq, C_eq, D_eq
        A2, B2, C2, D2 = row['A_5'], row['B_5'], row['C_5'], row['D_5']

        A_eq = (A1 * B2 + A2 * B1) / (B1 + B2)
        B_eq = (B1 * B2) / (B1 + B2)
        C_eq = C1 + C2 + ((A1 - A2) * (D1 - D2)) / (B1 + B2)
        D_eq = (D1 * B2 + D2 * B1) / (B1 + B2)

    return pd.Series({
        'A': A_eq, 'B': B_eq, 'C': C_eq, 'D': D_eq
    })

# Combine lines with same (From, To)
ABCD_combined = (
    ABCD.groupby(['From Bus Number', 'To Bus Number'])
    .apply(combine_parallel)
    .reset_index()  # REMOVED drop=True so Pandas auto-restores the From/To columns
)

# Save result
ABCD_combined.to_csv("ABCD_parallel_combined_5.csv", index=False)

print(ABCD_combined.shape)

(27, 6)


C:\Users\User\AppData\Local\Temp\ipykernel_2764\2599097321.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(combine_parallel)


In [12]:
import pandas as pd
import numpy as np

# Load data
ABCD_combined = pd.read_csv("ABCD_parallel_combined.csv")

def abcd2s(df):
    S_data = []

    Z0 = 360.2607461227684
    Y0 = 1 / Z0

    for _, row in df.iterrows():
        from_bus = row["From Bus Number"]
        to_bus   = row["To Bus Number"]

        A = complex(row["A"])
        B = complex(row["B"])
        C = complex(row["C"])
        D = complex(row["D"])

        denom = (A + B*Y0 + C*Z0 + D)

        S11 = (A + B*Y0 - C*Z0 - D) / denom
        S12 = 2 * (A*D - B*C) / denom
        S21 = 2 / denom
        S22 = (-A + B*Y0 - C*Z0 + D) / denom

        S = np.array([
            [S11, S12],
            [S21, S22]
        ], dtype=complex)

        S_data.append({
            "from_bus": from_bus,
            "to_bus": to_bus,
            "S": S
        })

    return S_data
S_params = abcd2s(ABCD_combined)

for i, item in enumerate(S_params):
    print(f"\nLine {i}: Bus {item['from_bus']} → Bus {item['to_bus']}")
    print(item["S"])



Line 0: Bus 2135 → Bus 2220
[[-0.02353949-0.13114762j  0.97448622-0.17818419j]
 [ 0.97448622-0.17818419j -0.02353949-0.13114762j]]

Line 1: Bus 2135 → Bus 2400
[[-0.54056824-0.33097366j  0.3956228 -0.65689528j]
 [ 0.3956228 -0.65689528j -0.54056824-0.33097366j]]

Line 2: Bus 2135 → Bus 2970
[[-0.11785927-0.27322296j  0.8735043 -0.38158973j]
 [ 0.8735043 -0.38158973j -0.11785927-0.27322296j]]

Line 3: Bus 2220 → Bus 2225
[[-0.00822655-0.07379317j  0.98858779-0.12338457j]
 [ 0.98858779-0.12338457j -0.00822655-0.07379317j]]

Line 4: Bus 2220 → Bus 2230
[[-0.04405298-0.17722816j  0.9523104 -0.2411257j ]
 [ 0.9523104 -0.2411257j  -0.04405298-0.17722816j]]

Line 5: Bus 2220 → Bus 2570
[[-0.20462834-0.33361586j  0.78019381-0.48381641j]
 [ 0.78019381-0.48381641j -0.20462834-0.33361586j]]

Line 6: Bus 2220 → Bus 2691
[[-0.25270185-0.29857033j  0.68766833-0.60061615j]
 [ 0.68766833-0.60061615j -0.25270185-0.29857033j]]

Line 7: Bus 2222 → Bus 2300
[[-0.01609175-0.12395725j  0.98383294-0.1279877